In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import os

In [ ]:
spark = (
    SparkSession.builder
    .appName('pagila-tasks')
    .config('spark.jars.packages', 'org.postgresql:postgresql:42.7.3')
    .config('spark.sql.shuffle.partitions', '8')  # custom
    .getOrCreate()
)

In [ ]:
jdbc_url = f'jdbc:postgresql://db:5432/{os.environ['DB_NAME']}'
props = {
    'user': os.environ['DB_USER'],
    'password': os.environ['DB_PASSWORD'],
    'driver': 'org.postgresql.Driver',
}

In [ ]:
def read_table(table, columns=None, where=None):
    '''
    move filtering to the postgres level
    '''
    cols = ', '.join(columns) if columns else '*'
    query = f'(SELECT {cols} FROM {table}' + (f' WHERE {where}' if where else '') + ') AS t'
    return spark.read.jdbc(url=jdbc_url, table=query, properties=props)

Output the number of movies in each category, sorted in descending order

In [ ]:
film_category = read_table('film_category', ['film_id', 'category_id'])
category = read_table('category', ['category_id', 'name'])

res1 = (
    film_category.join(F.broadcast(category), 'category_id')
    .groupBy('name')
    .agg(F.count('category_id').alias('movie_count'))
    .orderBy(F.desc('movie_count'))
)
res1.show()

+-----------+-----------+
|       name|movie_count|
+-----------+-----------+
|      Music|        152|
|      Drama|        152|
|     Travel|        151|
|      Games|        150|
|    Foreign|        150|
|   Children|        150|
|     Sci-Fi|        149|
|     Action|        149|
|  Animation|        148|
|     Family|        147|
|   Classics|        147|
|        New|        147|
|Documentary|        145|
|     Sports|        145|
|     Comedy|        143|
|     Horror|        142|
+-----------+-----------+



In [ ]:
res1.explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Sort (10)
   +- Exchange (9)
      +- HashAggregate (8)
         +- Exchange (7)
            +- HashAggregate (6)
               +- Project (5)
                  +- BroadcastHashJoin Inner BuildRight (4)
                     :- Scan JDBCRelation((SELECT film_id, category_id FROM film_category) AS t) [numPartitions=1]  (1)
                     +- BroadcastExchange (3)
                        +- Scan JDBCRelation((SELECT category_id, name FROM category) AS t) [numPartitions=1]  (2)


(1) Scan JDBCRelation((SELECT film_id, category_id FROM film_category) AS t) [numPartitions=1] 
Output [1]: [category_id#1]
PushedFilters: [*IsNotNull(category_id)]
ReadSchema: struct<category_id:int>

(2) Scan JDBCRelation((SELECT category_id, name FROM category) AS t) [numPartitions=1] 
Output [2]: [category_id#4, name#5]
PushedFilters: [*IsNotNull(category_id)]
ReadSchema: struct<category_id:int,name:string>

(3) BroadcastExchange
Input [2]: [category_id#4, na

Output the 10 actors whose movies rented the most, sorted in descending order. 

In [ ]:
actor = read_table('actor', ['actor_id', 'first_name', 'last_name'])
film_actor = read_table('film_actor', ['actor_id', 'film_id'])
inventory = read_table('inventory', ['inventory_id', 'film_id'])
rental = read_table('rental', ['inventory_id'])

res2 = (
    rental
    .join(inventory, 'inventory_id')
    .join(film_actor, 'film_id')
    .groupBy('actor_id')
    .agg(F.count('*').alias('rental_num'))
    .join(F.broadcast(actor), 'actor_id')
    .orderBy(F.desc('rental_num'))
    .limit(10)
)
res2.show()

+--------+----------+----------+-----------+
|actor_id|rental_num|first_name|  last_name|
+--------+----------+----------+-----------+
|     107|      2426|      GINA|  DEGENERES|
|     181|      2247|   MATTHEW|     CARREY|
|     198|      2173|      MARY|     KEITEL|
|     144|      2146|    ANGELA|WITHERSPOON|
|     102|      2074|    WALTER|       TORN|
|      37|      2012|       VAL|     BOLGER|
|     150|      2010|     JAYNE|      NOLTE|
|      60|      1978|     HENRY|      BERRY|
|      23|      1958|    SANDRA|     KILMER|
|      90|      1903|      SEAN|    GUINESS|
+--------+----------+----------+-----------+



In [ ]:
res2.explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (24)
+- TakeOrderedAndProject (23)
   +- Project (22)
      +- BroadcastHashJoin Inner BuildRight (21)
         :- HashAggregate (18)
         :  +- Exchange (17)
         :     +- HashAggregate (16)
         :        +- Project (15)
         :           +- SortMergeJoin Inner (14)
         :              :- Sort (10)
         :              :  +- Exchange (9)
         :              :     +- Project (8)
         :              :        +- SortMergeJoin Inner (7)
         :              :           :- Sort (3)
         :              :           :  +- Exchange (2)
         :              :           :     +- Scan JDBCRelation((SELECT inventory_id FROM rental) AS t) [numPartitions=1]  (1)
         :              :           +- Sort (6)
         :              :              +- Exchange (5)
         :              :                 +- Scan JDBCRelation((SELECT inventory_id, film_id FROM inventory) AS t) [numPartitions=1]  (4)
         :              

Output the category of movies on which the most money was spent

In [ ]:
sales_by_film_category = read_table('sales_by_film_category', ['category', 'total_sales'])
res3 = (
    sales_by_film_category
    .withColumn('rnk', F.rank().over(Window.orderBy(F.desc('total_sales'))))
    .filter(F.col('rnk') == 1)
    .select('category', F.round('total_sales', 2).alias('total_sales'))
)
res3.show()

+--------+-----------+
|category|total_sales|
+--------+-----------+
|  Action|   26505.44|
+--------+-----------+



In [ ]:
res3.explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (10)
+- Project (9)
   +- Filter (8)
      +- Window (7)
         +- WindowGroupLimit (6)
            +- Sort (5)
               +- Exchange (4)
                  +- WindowGroupLimit (3)
                     +- Sort (2)
                        +- Scan JDBCRelation((SELECT category, total_sales FROM sales_by_film_category) AS t) [numPartitions=1]  (1)


(1) Scan JDBCRelation((SELECT category, total_sales FROM sales_by_film_category) AS t) [numPartitions=1] 
Output [2]: [category#337, total_sales#338]
ReadSchema: struct<category:string,total_sales:decimal(38,18)>

(2) Sort
Input [2]: [category#337, total_sales#338]
Arguments: [total_sales#338 DESC NULLS LAST], false, 0

(3) WindowGroupLimit
Input [2]: [category#337, total_sales#338]
Arguments: [total_sales#338 DESC NULLS LAST], rank(total_sales#338), 1, Partial

(4) Exchange
Input [2]: [category#337, total_sales#338]
Arguments: SinglePartition, ENSURE_REQUIREMENTS, [plan_id=1838]

(5) Sort
Input [2]:

Output the names of movies that are not in the inventory.

In [25]:
film = read_table('film', ['film_id', 'title'])
inv_ids = read_table('inventory', ['film_id'])

res4 = (
    film.join(inv_ids, 'film_id', 'left_anti').select("title")
)
res4.show()

+--------------------+
|               title|
+--------------------+
|      ALICE FANTASIA|
|       ARK RIDGEMONT|
|      CHOCOLATE DUCK|
|COMMANDMENTS EXPRESS|
|    CRYSTAL BREAKING|
|DELIVERANCE MULHO...|
|         SKY MIRACLE|
|   RAIDERS ANTITRUST|
|       ROOF CHAMPION|
|    SUICIDES SILENCE|
|       BUTCH PANTHER|
|     CROWDS TELEMARK|
|       SISTER FREDDY|
|   VILLAIN DESPERATE|
|        VOLUME HOUSE|
|         APOLLO TEEN|
|ARSENIC INDEPENDENCE|
|          DAZED PUNK|
|FRANKENSTEIN STRA...|
|      ORDER BETRAYED|
+--------------------+
only showing top 20 rows



In [26]:
res4.explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- SortMergeJoin LeftAnti (7)
      :- Sort (3)
      :  +- Exchange (2)
      :     +- Scan JDBCRelation((SELECT film_id, title FROM film) AS t) [numPartitions=1]  (1)
      +- Sort (6)
         +- Exchange (5)
            +- Scan JDBCRelation((SELECT film_id FROM inventory) AS t) [numPartitions=1]  (4)


(1) Scan JDBCRelation((SELECT film_id, title FROM film) AS t) [numPartitions=1] 
Output [2]: [film_id#391, title#392]
ReadSchema: struct<film_id:int,title:string>

(2) Exchange
Input [2]: [film_id#391, title#392]
Arguments: hashpartitioning(film_id#391, 8), ENSURE_REQUIREMENTS, [plan_id=2095]

(3) Sort
Input [2]: [film_id#391, title#392]
Arguments: [film_id#391 ASC NULLS FIRST], false, 0

(4) Scan JDBCRelation((SELECT film_id FROM inventory) AS t) [numPartitions=1] 
Output [1]: [film_id#395]
PushedFilters: [*IsNotNull(film_id)]
ReadSchema: struct<film_id:int>

(5) Exchange
Input [1]: [film_id#395]
Arguments: hashpartitioning